# Attention Mechanism — Exercises

Companion to the note [Attention Mechanism](Attention%20Mechanism.md).

Practise the Q/K/V roles, masking, cross-attention, batched multi-head attention, grouped-query attention, the KV-cache, additive attention, the online softmax behind FlashAttention and the backward pass. Everything is built from scratch in NumPy and checked against SciPy references, property tests and finite differences.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×4 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.

In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax as sp_softmax
rng = np.random.default_rng(3)

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f() w.r.t. array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Queries, keys, values and the three flavours
*🧠 Concept · ★☆☆*

Using the library analogy of the note, explain what the **query**, **key** and **value** of a token are, and why keys and values are separate projections of the same input.
Then fill in the table: for self-attention, cross-attention and causal self-attention, where do $Q$ and $K,V$ come from?
If $Q$ has $m$ rows and $K,V$ have $n$ rows, what are the shapes of the score matrix and of the output?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Matching and handing over content are different jobs. In cross-attention the two sequences can have different lengths.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Query = what a token looks for, key = what it advertises for matching, value = what it contributes if selected. Separate projections let a token be *found* by one feature
and *deliver* another (a verb may be matched by its tense but hand over its meaning).

| Kind | $Q$ from | $K,V$ from |
|---|---|---|
| self-attention | $X$ | the same $X$ |
| cross-attention | decoder states | encoder outputs |
| causal self-attention | $X$ | only positions $\le i$ of $X$ |

Scores are $m\times n$ (one row per query), the output is $m\times d_v$: **one output per query**, whatever $n$ is.

</details>

### Exercise 2 · Hard lookup, soft lookup, and the RNN bottleneck
*🧠 Concept · ★☆☆*

1. Why is attention called a *soft* lookup, and why does that matter for training?
2. What problem of encoder–decoder RNNs did Bahdanau attention (2014) fix?
3. Compare additive attention $e_{ij}=v^\top\tanh(W_sh_{i-1}+W_hh_j)$ with dot-product attention. Which is cheaper, and why do Transformers use the dot-product form?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A hard lookup is an argmax. Think about its gradient. For (3) think of what runs well as one big matrix multiplication.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Every value contributes with a softmax weight, so the output is a differentiable function of queries and keys and can be trained with [[Backpropagation]]; an argmax has zero gradient almost everywhere.
2. The RNN had to squeeze the whole source sentence into **one fixed-size hidden state**. Attention lets each decoding step look back at *all* encoder states and weight them.
3. Additive attention runs a small MLP for every (query, key) pair. Dot-product attention is just $QK^\top$, one highly optimised matrix multiplication on a GPU, and costs no extra parameters.
   Both work about equally well for small $d_k$; the $1/\sqrt{d_k}$ scaling fixes dot-product attention for large $d_k$.

</details>

### Exercise 3 · Masks: where and when
*🧠 Concept · ★☆☆*

A colleague implements causal masking as `weights = softmax(S) * mask` (zeroing future weights **after** the softmax). What is wrong with this?
Write the correct version. Which two situations need a mask, and what is the typical shape of the mask for a padded batch?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Check whether each row of the weights still sums to 1. And does the softmax denominator still contain the future tokens?

</details>

<details>
<summary><b>✅ Solution</b></summary>

After zeroing, the rows no longer sum to 1, and, more importantly, the softmax *denominator* still included the forbidden positions, so the weights are distorted by tokens that should be invisible.
Correct: $A=\mathrm{softmax}(S/\sqrt{d_k}+M)$ with $M_{ij}=-\infty$ for forbidden pairs, i.e. `S = np.where(mask, S, -np.inf)` **before** the softmax; then those weights are exactly 0 and the remaining ones renormalise.

Masks are needed for **causality** (decoder: lower-triangular $n\times n$) and for **padding** (ignore padded keys: a boolean $B\times1\times n$ key mask broadcast over queries).

</details>

## Part B · By hand

### Exercise 4 · Causal attention by hand
*✍️ By hand · ★★☆*

Three tokens with $Q=K=V=X=\begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$ and $d_k=2$. Compute the **causal** self-attention weights $A$ (3×3, lower triangular) and the output $AV$ (3 decimals).

In [ ]:
X_h = np.array([[1, 0], [0, 1], [1, 1.]])
A_hand = None    # 3x3
out_hand = None  # 3x2
_S = X_h @ X_h.T / np.sqrt(2)
_A = sp_softmax(np.where(np.tril(np.ones((3, 3), bool)), _S, -np.inf), axis=1)
check('causal weights', A_hand, _A, tol=1e-3); check('output', out_hand, _A @ X_h, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Row $i$ only has $i+1$ allowed entries. Row 1 is $(1,0,0)$. Scores are $q_i\cdot k_j/\sqrt2$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Row 3: scores $(1,1,2)/\sqrt2=(0.707,0.707,1.414)$; exponentiate and normalise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$S/\sqrt2$ restricted to the allowed entries: row 1 $=(\cdot)$ → weight $1$; row 2 $=(0,\,0.707)$ → $(0.330,\,0.670)$; row 3 $=(0.707,0.707,1.414)$ → $e^{\cdot}=(2.028,2.028,4.113)$, sum $8.170$ → $(0.248,\,0.248,\,0.503)$.

Outputs: $o_1=(1,0)$; $o_2=0.330(1,0)+0.670(0,1)=(0.330,\,0.670)$; $o_3=0.248(1,0)+0.248(0,1)+0.503(1,1)=(0.752,\,0.752)$.

```python
X_h = np.array([[1, 0], [0, 1], [1, 1.]])
A_hand = [[1, 0, 0], [0.3302, 0.6698, 0], [0.2483, 0.2483, 0.5035]]
out_hand = [[1, 0], [0.3302, 0.6698], [0.7518, 0.7518]]
```

</details>

### Exercise 5 · Shapes and memory of one layer
*✍️ By hand · ★★☆*

A layer has $d=768$, $h=12$ heads, batch $B=8$ and $n=512$ tokens, in fp16 (2 bytes). Compute: the head size $d_k$, the number of entries of the score tensor $B\times h\times n\times n$,
its size in MiB, and the parameter count of the attention layer ($W_Q,W_K,W_V,W_O$, each $d\times d$ **with** a bias).

In [ ]:
d_k = None
score_elems = None
score_mib = None
params_attn = None
check('d_k', d_k, 64); check('score entries', score_elems, 8 * 12 * 512 * 512)
check('score MiB', score_mib, 8 * 12 * 512 * 512 * 2 / 2**20); check('parameters', params_attn, 4 * (768 * 768 + 768))

<details>
<summary><b>💡 Hint</b></summary>

Entries: $8\cdot12\cdot512\cdot512$. MiB $=$ entries $\cdot2/2^{20}$. Parameters: $4(d^2+d)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$d_k=768/12=64$. Score entries $8\cdot12\cdot512^2=25{,}165{,}824$; times 2 bytes $=48$ MiB **per layer, just for the scores** (this is what FlashAttention avoids writing out).
Parameters $4(768^2+768)=2{,}362{,}368$, independent of $h$.

```python
d_k = 64
score_elems = 25165824
score_mib = 48
params_attn = 2362368
```

</details>

### Exercise 6 · KV-cache with grouped-query attention
*✍️ By hand · ★★☆*

A decoder has $L=32$ layers, $h=32$ query heads of size $d_{head}=128$, context $n=4096$ and fp16 weights. Compare the KV-cache of full multi-head attention (32 KV heads) with
grouped-query attention with $g=8$ KV heads. Give the bytes per token and the GiB per sequence for GQA, and the saving factor.

In [ ]:
kv_bytes_token = None
kv_gib = None
saving = None
check('bytes per token (GQA)', kv_bytes_token, 2 * 32 * 8 * 128 * 2); check('GiB per sequence (GQA)', kv_gib, 2 * 32 * 8 * 128 * 2 * 4096 / 2**30)
check('saving vs MHA', saving, 32 / 8)

<details>
<summary><b>💡 Hint</b></summary>

Per layer and token you cache $K$ and $V$: $2\cdot g\cdot d_{head}$ numbers.

</details>

<details>
<summary><b>✅ Solution</b></summary>

GQA: $2\,(K,V)\cdot32\cdot8\cdot128\cdot2\text{ B}=131{,}072$ B $=128$ KiB per token; times 4096 tokens $=512$ MiB $=0.5$ GiB per sequence.
Full MHA stores 32 instead of 8 heads: $2$ GiB. Saving factor $h/g=4$ with very little quality loss, which is why Llama 2/3 use it.

```python
kv_bytes_token = 131072
kv_gib = 0.5
saving = 4
```

</details>

### Exercise 7 · Temperature of the softmax
*✍️ By hand · ★☆☆*

For scores $s=(2,1,0)$ compute the attention weights $\mathrm{softmax}(s)$ and $\mathrm{softmax}(s/T)$ with $T=0.5$ (3 decimals), and the entropy $H=-\sum_ip_i\ln p_i$ of the first one (in nats).
What happens to the weights as $T\to0$ and as $T\to\infty$? How is $\sqrt{d_k}$ related to a temperature?

In [ ]:
s_t = np.array([2, 1, 0.])
w_T1 = None
w_T05 = None
entropy = None
_p = sp_softmax(s_t)
check('softmax(s)', w_T1, _p, tol=1e-3); check('softmax(s/0.5)', w_T05, sp_softmax(s_t / 0.5), tol=1e-3)
check('entropy (nats)', entropy, -np.sum(_p * np.log(_p)), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Exponentials: $e^2=7.389,\ e^1=2.718,\ e^0=1$. For $T=0.5$ use $(4,2,0)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{softmax}(2,1,0)=(0.665,\,0.245,\,0.090)$; $H=0.665\cdot0.408+0.245\cdot1.408+0.090\cdot2.408\approx0.832$ (maximum $\ln3=1.099$).
With $T=0.5$: $(4,2,0)\to(0.867,\,0.117,\,0.016)$, sharper. $T\to0$: one-hot (hard argmax); $T\to\infty$: uniform $1/3$ (output = mean of the values).
Dividing by $\sqrt{d_k}$ is a **fixed temperature** $T=\sqrt{d_k}$ that keeps the score scale constant across head sizes.

```python
s_t = np.array([2, 1, 0.])
w_T1 = [0.665, 0.245, 0.090]
w_T05 = [0.867, 0.117, 0.016]
entropy = 0.832
```

</details>

## Part C · Code from scratch

### Exercise 8 · Stable, masked, batched attention
*💻 Code · ★☆☆*

Implement `softmax(S, axis=-1)` (no overflow for large inputs) and `attention(Q, K, V, mask=None)` returning `(output, weights)`.
It must work on **batches**: `Q` has shape `(..., m, d_k)`, `K` `(..., n, d_k)`, `V` `(..., n, d_v)`; `mask` is a boolean array broadcastable to `(..., m, n)`, `True` where attention is **allowed**.
Test it with a padding mask: in a batch of two sequences with lengths 6 and 4, the padded keys of the second sequence must get weight exactly 0, and the result must equal attention on the un-padded sequence.

In [ ]:
def softmax(S, axis=-1):
    return None

def attention(Q, K, V, mask=None):
    return None, None
Qb, Kb, Vb = rng.normal(size=(2, 4, 8)), rng.normal(size=(2, 6, 8)), rng.normal(size=(2, 6, 3))
lengths = np.array([6, 4])
pad_mask = (np.arange(6)[None, :] < lengths[:, None])[:, None, :]        # (B, 1, n)
out_b, A_b = attention(Qb, Kb, Vb, pad_mask)
out_u, _ = attention(Qb[1], Kb[1, :4], Vb[1, :4])

check('no overflow', softmax(np.array([1000., 1001., 1002.])), sp_softmax([1000., 1001., 1002.]))
check('batched output', out_b if out_b is None else out_b[0], sp_softmax(Qb[0] @ Kb[0].T / np.sqrt(8), axis=1) @ Vb[0])
check('padded keys get weight 0', None if A_b is None else np.abs(A_b[1, :, 4:]).max(), 0.0)
check('rows still sum to 1', None if A_b is None else A_b.sum(-1), np.ones((2, 4)))
check('padded batch == unpadded sequence', None if out_b is None else out_b[1], out_u)

<details>
<summary><b>💡 Hint</b></summary>

Subtract the row max before `np.exp`. Use `np.swapaxes(K, -1, -2)` for the transpose so that leading batch axes are untouched. Padding mask: `np.arange(n) < length` has shape $B\times n$; add an axis for the query dimension.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Subtracting the row max leaves the softmax unchanged but makes the largest exponent $e^0=1$. Setting masked scores to $-\infty$ gives them weight exactly $0$ and the others renormalise;
the padded batch therefore gives the same result as running each sequence separately.

```python
def softmax(S, axis=-1):
    S = S - S.max(axis=axis, keepdims=True)
    E = np.exp(S)
    return E / E.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    S = Q @ np.swapaxes(K, -1, -2) / np.sqrt(Q.shape[-1])
    if mask is not None:
        S = np.where(mask, S, -np.inf)
    A = softmax(S, axis=-1)
    return A @ V, A
```

</details>

### Exercise 9 · Cross-attention
*💻 Code · ★★☆*

Implement `cross_attention(X_dec, H_enc, Wq, Wk, Wv)`: queries come from the decoder states `X_dec` $(m\times d)$, keys and values from the encoder states `H_enc` $(n\times d)$. Return `(out, A)`.
Verify the two defining properties: each output row depends only on **its own** query row, but **every** output row changes when an encoder state changes.

In [ ]:
def cross_attention(X_dec, H_enc, Wq, Wk, Wv):
    return None, None
d_x, m_x, n_x, dk_x, dv_x = 8, 3, 5, 4, 6
X_dec, H_enc = rng.normal(size=(m_x, d_x)), rng.normal(size=(n_x, d_x))
Wq_x, Wk_x, Wv_x = rng.normal(size=(d_x, dk_x)), rng.normal(size=(d_x, dk_x)), rng.normal(size=(d_x, dv_x))
out_x, A_x = cross_attention(X_dec, H_enc, Wq_x, Wk_x, Wv_x)
_A = sp_softmax((X_dec @ Wq_x) @ (H_enc @ Wk_x).T / np.sqrt(dk_x), axis=1)
check('weights are m x n and match', A_x, _A); check('output is m x d_v', None if out_x is None else out_x, _A @ (H_enc @ Wv_x))
X2 = X_dec.copy(); X2[1] = rng.normal(size=d_x)
H2 = H_enc.copy(); H2[2] = rng.normal(size=d_x)
o_q, _ = cross_attention(X2, H_enc, Wq_x, Wk_x, Wv_x)
o_h, _ = cross_attention(X_dec, H2, Wq_x, Wk_x, Wv_x)
check('changing query 1 changes only output row 1', None if o_q is None else np.abs(o_q - out_x).max(1) > 1e-9, [False, True, False])
check('changing an encoder state changes every row', None if o_h is None else np.abs(o_h - out_x).max(1) > 1e-9, [True, True, True])

<details>
<summary><b>💡 Hint</b></summary>

Reuse `attention` from the previous exercise: `Q = X_dec @ Wq`, `K = H_enc @ Wk`, `V = H_enc @ Wv`. The weights have shape $m\times n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Cross-attention is just attention with $Q$ and $K,V$ computed from different inputs, so the score matrix is rectangular, $m\times n$. This is how a translation decoder consults the source sentence
or how a text-to-image model reads the prompt. Queries are processed independently of each other (row $i$ of the output depends only on query $i$), but all of them read the same keys and values.

```python
def cross_attention(X_dec, H_enc, Wq, Wk, Wv):
    return attention(X_dec @ Wq, H_enc @ Wk, H_enc @ Wv)
```

</details>

### Exercise 10 · Batched multi-head attention with reshapes
*💻 Code · ★★☆*

Implement `mha(X, Wq, Wk, Wv, Wo, h, mask=None)` for `X` of shape `(B, n, d)` **without a Python loop over heads**: project, reshape to `(B, h, n, d/h)` (`reshape` then `transpose`),
call `attention`, merge the heads back to `(B, n, d)` and multiply by `Wo`. Return `(out, A)` with `A` of shape `(B, h, n, n)`. Check against a loop-based reference and with a causal mask.

In [ ]:
def mha(X, Wq, Wk, Wv, Wo, h, mask=None):
    return None, None
B_m, n_m, d_m, h_m = 3, 5, 8, 4
X_m = rng.normal(size=(B_m, n_m, d_m))
Wq_m, Wk_m, Wv_m, Wo_m = (rng.normal(0, d_m ** -0.5, size=(d_m, d_m)) for _ in range(4))
def _mha_loop(X, h):
    dk = X.shape[2] // h; Q, K, V = X @ Wq_m, X @ Wk_m, X @ Wv_m; res = []
    for b in range(X.shape[0]):
        heads = [sp_softmax(Q[b][:, j*dk:(j+1)*dk] @ K[b][:, j*dk:(j+1)*dk].T / np.sqrt(dk), axis=1) @ V[b][:, j*dk:(j+1)*dk] for j in range(h)]
        res.append(np.concatenate(heads, axis=1) @ Wo_m)
    return np.stack(res)
out_m, A_m = mha(X_m, Wq_m, Wk_m, Wv_m, Wo_m, h_m)
out_c, A_c = mha(X_m, Wq_m, Wk_m, Wv_m, Wo_m, h_m, mask=np.tril(np.ones((n_m, n_m), bool)))
check('matches the per-head loop', out_m, _mha_loop(X_m, h_m))
check('weights shape (B, h, n, n)', None if A_m is None else A_m.shape, (3, 4, 5, 5))
check('causal: no weight on the future', None if A_c is None else np.abs(np.triu(A_c, k=1)).max(), 0.0)

<details>
<summary><b>💡 Hint</b></summary>

`Q.reshape(B, n, h, dk).transpose(0, 2, 1, 3)` gives `(B, h, n, dk)`. To merge: `out.transpose(0, 2, 1, 3).reshape(B, n, d)`. The mask `(n, n)` broadcasts over `(B, h, n, n)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The reshape trick turns "$h$ separate attentions" into **one** batched matrix multiplication, which is how every deep-learning library does it. Heads share the input but have different column slices of $W_Q,W_K,W_V$.
Because `attention` already accepts leading batch axes, nothing else changes.

```python
def mha(X, Wq, Wk, Wv, Wo, h, mask=None):
    B, n, d = X.shape
    dk = d // h
    split = lambda M: M.reshape(B, n, h, dk).transpose(0, 2, 1, 3)       # (B, h, n, dk)
    out, A = attention(split(X @ Wq), split(X @ Wk), split(X @ Wv), mask)
    return out.transpose(0, 2, 1, 3).reshape(B, n, d) @ Wo, A
```

</details>

### Exercise 11 · Additive (Bahdanau) attention
*💻 Code · ★★☆*

Implement `additive_attention(s, H, Ws, Wh, v)` for one decoder state `s` $(d_s)$ and encoder states `H` $(n\times d_h)$:
$e_j=v^\top\tanh(W_ss+W_hh_j)$, $\alpha=\mathrm{softmax}(e)$, context $c=\sum_j\alpha_jh_j$. Return `(c, alpha)`.
Sanity-check: with $v=0$ all scores are equal, so $\alpha$ is uniform and $c$ is the mean of the encoder states.

In [ ]:
def additive_attention(s, H, Ws, Wh, v):
    return None, None
ds, dh, da, n_a = 4, 6, 5, 7
s_a, H_a = rng.normal(size=ds), rng.normal(size=(n_a, dh))
Ws_a, Wh_a, v_a = rng.normal(size=(da, ds)), rng.normal(size=(da, dh)), rng.normal(size=da)
c_a, al_a = additive_attention(s_a, H_a, Ws_a, Wh_a, v_a)
_e = np.array([v_a @ np.tanh(Ws_a @ s_a + Wh_a @ H_a[j]) for j in range(n_a)])
check('attention weights', al_a, sp_softmax(_e)); check('context vector', c_a, sp_softmax(_e) @ H_a)
c0, al0 = additive_attention(s_a, H_a, Ws_a, Wh_a, np.zeros(da))
check('v = 0 -> uniform weights', al0, np.full(n_a, 1 / n_a)); check('v = 0 -> mean of the encoder states', c0, H_a.mean(0))

<details>
<summary><b>💡 Hint</b></summary>

Vectorise over $j$: `np.tanh(Ws @ s + H @ Wh.T)` has shape $(n, d_a)$; multiply by `v` to get the $n$ scores.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The score comes from a one-hidden-layer network applied to every pair $(s,h_j)$. Compared with $q\cdot k$ it has extra parameters and a tanh per pair; Luong (2015) showed the plain dot product (or a bilinear form) is simpler and almost as good, and it maps onto one matrix multiplication.

```python
def additive_attention(s, H, Ws, Wh, v):
    e = np.tanh(Ws @ s + H @ Wh.T) @ v
    alpha = sp_softmax(e)
    return alpha @ H, alpha
```

</details>

### Exercise 12 · Incremental decoding with a KV cache
*💻 Code · ★★☆*

Implement `decode_with_cache(X, Wq, Wk, Wv)`: process the rows of `X` **one token at a time** as in generation. For token $t$ compute only its own $q_t,k_t,v_t$, append $k_t,v_t$ to the cache and attend over the cache. Return the stacked outputs `(n, d_v)`.
It must equal full causal self-attention. Then compute, for $n=6$ tokens, the number of query–key dot products with the cache (`dots_cached`) and when each step recomputes all scores from scratch (`dots_naive`).

In [ ]:
def decode_with_cache(X, Wq, Wk, Wv):
    return None

dots_cached = None
dots_naive = None
X_k = rng.normal(size=(6, 8)); Wq_k, Wk_k, Wv_k = (rng.normal(size=(8, 4)) for _ in range(3))
_Q, _K, _V = X_k @ Wq_k, X_k @ Wk_k, X_k @ Wv_k
_ref = sp_softmax(np.where(np.tril(np.ones((6, 6), bool)), _Q @ _K.T / 2, -np.inf), axis=1) @ _V
check('cached decoding == full causal attention', decode_with_cache(X_k, Wq_k, Wk_k, Wv_k), _ref)
check('dot products with cache', dots_cached, 21); check('dot products without cache', dots_naive, 91)

<details>
<summary><b>💡 Hint</b></summary>

Keep Python lists `Kc`, `Vc`. At step $t$ the new query is compared with $t+1$ keys. Without a cache, step $t$ recomputes a $(t+1)\times(t+1)$ score matrix.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With a cache, step $t$ costs $t+1$ dot products, in total $\sum_{t=1}^{n}t=n(n+1)/2=21$ for $n=6$. Recomputing everything each step costs $\sum_{t=1}^{n}t^2=n(n+1)(2n+1)/6=91$.
Both give identical outputs (the past never changes thanks to the causal mask), so the cache trades **memory** (Exercise 6) for an order of magnitude less compute: $O(n)$ instead of $O(n^2)$ per generated token.

```python
def decode_with_cache(X, Wq, Wk, Wv):
    Kc, Vc, outs = [], [], []
    for x in X:
        q, k, v = x @ Wq, x @ Wk, x @ Wv
        Kc.append(k); Vc.append(v)
        w = sp_softmax(np.array(Kc) @ q / np.sqrt(len(q)))
        outs.append(w @ np.array(Vc))
    return np.array(outs)

dots_cached = sum(t for t in range(1, 7))
dots_naive = sum(t * t for t in range(1, 7))
```

</details>

### Exercise 13 · Grouped-query attention
*💻 Code · ★★★*

Implement `gqa(X, Wq, Wk, Wv, Wo, h, g)` for one sequence `X` $(n\times d)$: $h$ query heads of size $d_k=d/h$, but only $g$ key/value heads (`Wk`, `Wv` have shape $d\times g\,d_k$). Query head $j$ uses KV head $\lfloor j/(h/g)\rfloor$.
Check against a reference that repeats the KV heads, and verify the multi-query case $g=1$ with a property test: if all query heads are identical, all head outputs must be identical.

In [ ]:
def gqa(X, Wq, Wk, Wv, Wo, h, g):
    return None
n_g, d_g, h_g = 5, 8, 4
X_g = rng.normal(size=(n_g, d_g)); Wq_g, Wo_g = rng.normal(size=(d_g, d_g)), rng.normal(size=(d_g, d_g))
def _mha_ref(X, Wq, Wk, Wv, Wo, h):   # plain MHA, one head after another
    dk = X.shape[1] // h; Q, K, V = X @ Wq, X @ Wk, X @ Wv
    return np.concatenate([sp_softmax(Q[:, j*dk:(j+1)*dk] @ K[:, j*dk:(j+1)*dk].T / np.sqrt(dk), axis=1) @ V[:, j*dk:(j+1)*dk] for j in range(h)], axis=1) @ Wo
Wk_f, Wv_f = rng.normal(size=(d_g, d_g)), rng.normal(size=(d_g, d_g))
check('g = h is ordinary multi-head attention', gqa(X_g, Wq_g, Wk_f, Wv_f, Wo_g, h_g, h_g), _mha_ref(X_g, Wq_g, Wk_f, Wv_f, Wo_g, h_g))
Wk_1, Wv_1 = rng.normal(size=(d_g, 2)), rng.normal(size=(d_g, 2))      # g = 1 -> one KV head of size d_k = 2
Wq_same = np.tile(rng.normal(size=(d_g, 2)), (1, h_g))                 # identical query heads
o_mqa = gqa(X_g, Wq_same, Wk_1, Wv_1, np.eye(d_g), h_g, 1)
check('MQA: identical queries -> identical heads', None if o_mqa is None else o_mqa.reshape(n_g, h_g, 2).std(axis=1).max(), 0.0, tol=1e-9)
_o_g2 = gqa(X_g, Wq_g, rng.normal(size=(d_g, 4)), rng.normal(size=(d_g, 4)), Wo_g, h_g, 2)
check('g = 2 runs and has shape (n, d)', None if _o_g2 is None else _o_g2.shape, (n_g, d_g))

<details>
<summary><b>💡 Hint</b></summary>

Reshape `K` to `(n, g, dk)` and use `np.repeat(K, h // g, axis=1)` to get `(n, h, dk)`; then every head can attend as in ordinary MHA.

</details>

<details>
<summary><b>✅ Solution</b></summary>

GQA interpolates between multi-head ($g=h$) and multi-query ($g=1$) attention: the number of KV heads sets the KV-cache size and the parameters of $W_K,W_V$, while queries keep full diversity.
The property test works because with $g=1$ all heads see the *same* $K,V$, so identical queries give identical outputs.

```python
def gqa(X, Wq, Wk, Wv, Wo, h, g):
    n, d = X.shape
    dk = d // h
    Q = (X @ Wq).reshape(n, h, dk)
    K = np.repeat((X @ Wk).reshape(n, g, dk), h // g, axis=1)
    V = np.repeat((X @ Wv).reshape(n, g, dk), h // g, axis=1)
    heads = [sp_softmax(Q[:, j] @ K[:, j].T / np.sqrt(dk), axis=1) @ V[:, j] for j in range(h)]
    return np.concatenate(heads, axis=1) @ Wo
```

</details>

### Exercise 14 · Online softmax: the idea behind FlashAttention
*💻 Code · ★★★*

FlashAttention never builds the $n\times n$ matrix; it streams over blocks of keys. For one query $q$ implement `online_attention(q, K, V, block)`, which processes $K,V$ in chunks of `block` rows while keeping
a running maximum $m$, a running normaliser $\ell$ and a running (unnormalised) output $a$:
$m'=\max(m,\max_j s_j)$, $\ell\leftarrow\ell\,e^{m-m'}+\sum_je^{s_j-m'}$, $a\leftarrow a\,e^{m-m'}+\sum_je^{s_j-m'}v_j$; the result is $a/\ell$. It must equal ordinary attention for every block size, even for huge scores.

In [ ]:
def online_attention(q, K, V, block):
    return None
q_o, K_o, V_o = rng.normal(size=16) * 5, rng.normal(size=(50, 16)) * 5, rng.normal(size=(50, 3))
_ref = sp_softmax(K_o @ q_o / 4) @ V_o
for blk in (1, 7, 16, 50):
    check(f'block = {blk}', online_attention(q_o, K_o, V_o, blk), _ref)
q_big = q_o * 100
check('huge scores do not overflow', online_attention(q_big, K_o, V_o, 8), sp_softmax(K_o @ q_big / 4) @ V_o)

<details>
<summary><b>💡 Hint</b></summary>

Start with $m=-\infty$ (then $e^{m-m'}=0$), $\ell=0$, $a=0$. Per block: `s = Kb @ q / sqrt(dk)`, `m_new = max(m, s.max())`, `scale = exp(m - m_new)`, `p = exp(s - m_new)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

When the maximum changes from $m$ to $m'$, everything accumulated so far was computed relative to the old maximum and is rescaled by $e^{m-m'}$, so after the last block $a/\ell=\sum_je^{s_j-m}v_j/\sum_je^{s_j-m}$ exactly.
Memory is $O(\text{block})$ instead of $O(n)$ per query (and $O(n)$ instead of $O(n^2)$ overall); the arithmetic is identical, only the schedule changes.

```python
def online_attention(q, K, V, block):
    m, l, acc = -np.inf, 0.0, np.zeros(V.shape[1])
    for i in range(0, len(K), block):
        s = K[i:i+block] @ q / np.sqrt(len(q))
        m_new = max(m, s.max())
        scale = np.exp(m - m_new)
        p = np.exp(s - m_new)
        l = l * scale + p.sum()
        acc = acc * scale + p @ V[i:i+block]
        m = m_new
    return acc / l
```

</details>

### Exercise 15 · Backward pass through attention
*💻 Code · ★★★*

The softmax Jacobian for one row is $\mathrm{diag}(a)-aa^\top$, so for upstream gradient $dA$: $dS=A\odot\big(dA-\sum_j dA_{ij}A_{ij}\big)$ (row-wise).
Using it, implement `attention_backward(Q, K, V, dO)` returning `dQ, dK, dV` for $O=\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ (no mask) and check all three with finite differences.

In [ ]:
def attention_backward(Q, K, V, dO):
    return None, None, None
Q_b, K_b, V_b, G_b = rng.normal(size=(4, 3)), rng.normal(size=(5, 3)), rng.normal(size=(5, 2)), rng.normal(size=(4, 2))
dQ_b, dK_b, dV_b = attention_backward(Q_b, K_b, V_b, G_b)
_f = lambda: np.sum((sp_softmax(Q_b @ K_b.T / np.sqrt(3), axis=1) @ V_b) * G_b)
check('dQ (finite differences)', dQ_b, num_grad(_f, Q_b), tol=1e-5)
check('dK (finite differences)', dK_b, num_grad(_f, K_b), tol=1e-5)
check('dV (finite differences)', dV_b, num_grad(_f, V_b), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Forward: $A=\mathrm{softmax}(S)$, $S=QK^\top/\sqrt{d_k}$. Backward: $dV=A^\top dO$, $dA=dO\,V^\top$, then $dS$ from the softmax formula, $dQ=dS\,K/\sqrt{d_k}$, $dK=dS^\top Q/\sqrt{d_k}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The chain rule through $O=AV$ gives $dV=A^\top dO$ and $dA=dO\,V^\top$. The row-wise softmax Jacobian is symmetric, $\mathrm{diag}(a)-aa^\top$, so $dS_{ij}=A_{ij}(dA_{ij}-\sum_kA_{ik}dA_{ik})$.
Finally $S=QK^\top/\sqrt{d_k}$ gives $dQ=dS\,K/\sqrt{d_k}$ and $dK=dS^\top Q/\sqrt{d_k}$. FlashAttention recomputes $A$ block by block in the backward pass instead of storing it.

```python
def attention_backward(Q, K, V, dO):
    A = sp_softmax(Q @ K.T / np.sqrt(Q.shape[1]), axis=1)
    dV = A.T @ dO
    dA = dO @ V.T
    dS = A * (dA - (dA * A).sum(axis=1, keepdims=True))
    return dS @ K / np.sqrt(Q.shape[1]), dS.T @ Q / np.sqrt(Q.shape[1]), dV
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Attention Mechanism](Attention%20Mechanism.md).*